# 07 - Validasi 5-seed + CI95 + uji signifikansi (Paper 4, Fase 3)

**Tujuan:** ulang pembandingan inti (centralized / local-only / federated-SFM, dan
FedAvg vs FedProx) pada **5 seed {13,42,101,202,303}**, laporkan **rerata +/- std +
CI95**, dan **uji signifikansi** (paired t-test / Wilcoxon) federated-SFM vs
local-only. Pola warisi Paper 2 (kejujuran statistik, bukan single-run).

**Konsistensi:** FedAvg via `../aws/fed_node.py`; yang berubah hanya SEED (init
bobot + urutan minibatch + partisi). Output: `fed_out/fed_agg.csv` (rerata/CI) +
`fed_out/significance_fed.csv`.

> Jalankan setelah nb01. Berat (5x training) -> jalankan di SageMaker. JANGAN mengarang.

## 0. Setup + impor fed_node

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scipy','scikit-learn'):
    if importlib.util.find_spec(m) is None and m!='scikit-learn':
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
    if m=='scikit-learn' and importlib.util.find_spec('sklearn') is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef
from scipy import stats
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
SEEDS=[13,42,101,202,303]; ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3
print('fed_node loaded | seeds', SEEDS)

## 1. Muat ruang SFM

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p,allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
Xc_tr,yc_tr=load('cic_train.npz'); Xc_te,yc_te=load('cic_test.npz')
Xu_tr,yu_tr=load('unsw_train.npz'); Xu_te,yu_te=load('unsw_test.npz')
ok=all(v is not None for v in [Xc_tr,Xu_tr,Xc_te,Xu_te])
if ok:
    Xg_te=np.vstack([Xc_te,Xu_te]); yg_te=np.concatenate([yc_te,yu_te])
    print('siap | GLOBAL-test', Xg_te.shape)
else:
    print('(partisi SFM belum ada - jalankan nb01)')

## 2. Fungsi training per-seed (federated-SFM & local-only)

Satu seed -> satu nilai MCC GLOBAL-test per setting. Dikumpulkan lintas 5 seed.

In [ ]:
def mcc_global(W, mu_g, sd_g):
    out,_=fed.forward(W, fed.zscore_apply(Xg_te,mu_g,sd_g)); return matthews_corrcoef(yg_te,(out>=0.5).astype(int))
def federated_sfm(seed):
    clients={'cic':(Xc_tr,yc_tr),'unsw':(Xu_tr,yu_tr)}
    zc={cid:fed.zscore_fit(X) for cid,(X,y) in clients.items()}
    Xz={cid:fed.zscore_apply(clients[cid][0],*zc[cid]) for cid in clients}
    sizes={cid:len(clients[cid][1]) for cid in clients}
    Wg=fed.init_weights(seed)
    for t in range(ROUNDS):
        cw=[];cs=[]
        for cid in clients:
            Wl=fed.train_local(Wg,Xz[cid],clients[cid][1],epochs=LOCAL_EPOCHS,lr=LR,mu=0.0,W_global=Wg,seed=seed)
            cw.append(Wl);cs.append(sizes[cid])
        Wg=fed.fedavg(cw,cs)
    mu_g,sd_g=fed.zscore_fit(np.vstack([Xc_tr,Xu_tr]))
    return mcc_global(Wg,mu_g,sd_g)
def local_only_best(seed):
    # lower bound: ambil MCC global TERBAIK antar model lokal (paling murah hati utk baseline)
    best=-1.0
    for cid,(Xl,yl) in {'cic':(Xc_tr,yc_tr),'unsw':(Xu_tr,yu_tr)}.items():
        mu_l,sd_l=fed.zscore_fit(Xl); Wl=fed.init_weights(seed)
        Wl=fed.train_local(Wl,fed.zscore_apply(Xl,mu_l,sd_l),yl,epochs=ROUNDS*LOCAL_EPOCHS,lr=LR,mu=0.0,seed=seed)
        best=max(best, mcc_global(Wl,mu_l,sd_l))
    return best
print('fungsi per-seed siap')

## 3. Jalankan 5 seed + agregasi (rerata/std/CI95)

In [ ]:
def ci95(a):
    a=np.asarray(a,float); m=a.mean(); s=a.std(ddof=1) if len(a)>1 else 0.0
    h=1.96*s/np.sqrt(len(a)) if len(a)>1 else 0.0
    return m, s, (m-h, m+h)
rows=[]; fed_scores=[]; loc_scores=[]
if ok:
    for sd in SEEDS:
        f=federated_sfm(sd); l=local_only_best(sd)
        fed_scores.append(f); loc_scores.append(l)
        rows.append({'seed':sd,'federated_SFM_MCC':round(f,4),'local_only_MCC':round(l,4)})
        print(f'seed {sd}: fed={f:.4f} local={l:.4f}')
    dfseed=pd.DataFrame(rows); dfseed.to_csv(os.path.join(OUTDIR,'fed_seeds.csv'),index=False)
    agg=[]
    for name,sc in [('federated-SFM',fed_scores),('local-only',loc_scores)]:
        m,s,(lo,hi)=ci95(sc)
        agg.append({'setting':name,'mean_MCC':round(m,4),'std':round(s,4),
                    'CI95_lo':round(lo,4),'CI95_hi':round(hi,4),'n_seed':len(sc)})
    dfagg=pd.DataFrame(agg); dfagg.to_csv(os.path.join(OUTDIR,'fed_agg.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfagg)
else:
    print('(lewati)')
print('=== SEL 3 (5-seed agregasi) SELESAI ===')

## 4. Uji signifikansi: federated-SFM vs local-only

In [ ]:
if ok and len(fed_scores)==len(loc_scores)==len(SEEDS):
    tt=stats.ttest_rel(fed_scores, loc_scores)
    try: ww=stats.wilcoxon(fed_scores, loc_scores)
    except Exception as e: ww=None; print('wilcoxon skip:',e)
    diff=np.array(fed_scores)-np.array(loc_scores)
    sig=[{'comparison':'federated-SFM vs local-only',
          'mean_diff':round(float(diff.mean()),4),
          'paired_t_stat':round(float(tt.statistic),4),'paired_t_p':round(float(tt.pvalue),5),
          'wilcoxon_p':(round(float(ww.pvalue),5) if ww is not None else None),
          'n_seed':len(SEEDS)}]
    dfsig=pd.DataFrame(sig); dfsig.to_csv(os.path.join(OUTDIR,'significance_fed.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfsig)
    print('Interpretasi: p<0.05 -> perbedaan federated vs local-only signifikan (lapor apa adanya).')
else:
    print('(uji signifikansi dilewati - butuh 5 seed lengkap)')
print('=== SEL 4 (signifikansi) SELESAI ===')

## 4b. Unggah SEMUA hasil `fed_out/` ke S3 (sekali jalan)

Mengumpulkan seluruh output run-2 dari nb01-nb07 yang ada di `fed_out/` lokal
(CSV + PNG + kurva) lalu meng-unggahnya ke
`s3://<bucket>/unsw-far/federated/results/run_2/`. Pakai **boto3** (bukan `aws s3 cp`,
pelajaran Paper 2). Setelah sel ini sukses, hasil bisa diunduh dari S3 untuk analisis
tanpa perlu re-run nb01-05. **Idempoten**: menimpa objek dengan nama sama.

In [ ]:
# Unggah semua artefak fed_out/ -> S3 results/run_2/ (boto3).
S3_BUCKET=os.environ.get('FED_S3_BUCKET','ssh-detection-features-232032302717')
REGION=os.environ.get('AWS_REGION','ap-southeast-1')
RUN_TAG=os.environ.get('FED_RUN_TAG','run_2')
PREFIX=f'unsw-far/federated/results/{RUN_TAG}'
try:
    import boto3
    s3=boto3.client('s3',region_name=REGION)
    exts=('.csv','.png','.json','.txt')
    files=sorted(f for f in os.listdir(OUTDIR) if f.endswith(exts))
    if not files:
        print('(tidak ada artefak di fed_out/ - pastikan nb01-07 sudah dijalankan)')
    up=0
    for f in files:
        key=f'{PREFIX}/{f}'
        s3.upload_file(os.path.join(OUTDIR,f), S3_BUCKET, key); up+=1
        print('  ok ->', f)
    print(f'\nunggah {up} berkas -> s3://{S3_BUCKET}/{PREFIX}/')
except Exception as e:
    print('(upload S3 gagal:',e,')')
print('=== SEL 4b (unggah hasil ke S3) SELESAI ===')

## 5. Catatan untuk naskah

- Hasil -> melengkapi SEMUA tabel (D-G) dgn rerata+/-std+CI95 (pola Paper 2).
- Uji signifikansi federated-SFM vs local-only: laporkan t-stat, p (t-test & Wilcoxon).
- local-only di sini pakai MCC global TERBAIK antar klien (baseline paling murah hati);
  jika federated tetap menang signifikan -> klaim kuat.
- Jujur: dgn 5 seed, Wilcoxon berdaya rendah; laporkan kedua uji + ukuran efek (mean_diff).
- Logika identik `aws/fed_node.py`; hanya seed berubah.